# MLflow Configuration et Training

## Setting up MLflow

In [1]:
# Install the required libraries
!pip install mlflow
!pip install --upgrade jinja2
!pip install --upgrade Flask
!pip install setuptools

  Using cached mlflow-3.10.1-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.10.1-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.6-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# starts an MLflow server locally.
!mlflow server --host 127.0.0.1 --port 8080

## Using the MLflow Client API

- Initiate a new Experiment.

- Start Runs within an Experiment.

- Document parameters, metrics, and tags for your Runs.

- Log artifacts linked to runs, such as models, tables, plots, and more.



In [2]:
from mlflow import MlflowClient


In [4]:
client = MlflowClient(tracking_uri="http://127.0.0.1:8080")


In [15]:
all_experiments = client.search_experiments()

print(all_experiments)

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1775439462065, experiment_id='0', last_update_time=1775439462065, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]


## Données et ingénierie des modèles

Ce notebook utilise le fichier **`data/data_clean_features.csv`** produit par `exploration_cleaning.ipynb` (prédiction de défaut de crédit, cible `default`).


Trois familles de modèles sont entraînées, chacune dans **son propre experiment** MLflow, avec itérations sur les hyperparamètres. Métriques journalisées : précision, rappel, F1, exactitude, ROC-AUC

In [21]:
import mlflow
import mlflow.sklearn
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [22]:
mlflow.set_tracking_uri("http://127.0.0.1:8080")

# Chemin projet : exécution depuis notebooks/ ou racine
_root = Path.cwd()
DATA_FILE = _root / "data" / "data_clean_features.csv"
if not DATA_FILE.is_file():
    DATA_FILE = _root.parent / "data" / "data_clean_features.csv"

df = pd.read_csv(DATA_FILE)
target_col = "default"
feature_cols = [c for c in df.columns if c != target_col]
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
def log_classification_run(experiment_name: str, run_name: str, model, params: dict):
    """Un run MLflow : entraînement, métriques, modèle sérialisé."""
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = {
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "precision": float(precision_score(y_test, y_pred, zero_division=0)),
            "recall": float(recall_score(y_test, y_pred, zero_division=0)),
            "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        }
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test)[:, 1]
            metrics["roc_auc"] = float(roc_auc_score(y_test, proba))
        mlflow.log_metrics(metrics)
        # Utiliser 'name' au lieu de 'artifact_path' (artifact_path est déprécié)
        mlflow.sklearn.log_model(model, name="model")
    return metrics

In [24]:
log_classification_run(
    experiment_name="loan_default_DecisionTree",
    run_name="dt_maxdepth_4",
    model=DecisionTreeClassifier(max_depth=4, random_state=42),
    params={"max_depth": 4, "min_samples_leaf": 1},
)

log_classification_run(
    experiment_name="loan_default_DecisionTree",
    run_name="dt_maxdepth_12_minleaf_5",
    model=DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=42),
    params={"max_depth": 12, "min_samples_leaf": 5},
)

2026/04/06 04:58:39 INFO mlflow.tracking.fluent: Experiment with name 'loan_default_DecisionTree' does not exist. Creating a new experiment.


2026/04/06 04:58:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run dt_maxdepth_4 at: http://127.0.0.1:8080/#/experiments/1/runs/013c8c27928e43468f1ab03ef24ae0db
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


2026/04/06 04:58:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run dt_maxdepth_12_minleaf_5 at: http://127.0.0.1:8080/#/experiments/1/runs/d9d3ee4582044a7e83c6df90d832bcba
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


{'accuracy': 0.993,
 'precision': 0.981081081081081,
 'recall': 0.981081081081081,
 'f1': 0.981081081081081,
 'roc_auc': 0.9971323163654453}